# 🤖 Notebook 7 — Pinecone for RAG (Read-Only)

> **No install required.** Walkthrough of building a **Retrieval-Augmented Generation** chatbot on Pinecone.

## 🎯 What you'll learn
- Why Pinecone is popular for production RAG
- The complete pipeline: chunk → embed → upsert → query → LLM answer
- **Namespaces** (multi-tenant isolation in one index)
- **Serverless vs Pod-based** indexes — when to pick which

## 🍱 Analogy
**Pinecone = AWS S3 for vectors.**
- Fully managed, pay-per-use, no infrastructure to operate.
- One API key, one URL, scales from 1k to 1B vectors.
- The price you pay for convenience: vendor lock-in and per-query cost.

In [ ]:
from IPython.display import SVG
SVG("""<svg viewBox="0 0 720 230" xmlns="http://www.w3.org/2000/svg" width="720"><style>text{font-family:Inter,system-ui,sans-serif}</style><rect width="720" height="230" fill="#f8fafc" rx="12"/><rect x="20" y="60" width="140" height="80" rx="10" fill="#dbeafe" stroke="#2563eb"/><text x="90" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">Docs</text><text x="90" y="110" text-anchor="middle" font-size="11" fill="#1e3a8a">PDFs / web</text><text x="90" y="125" text-anchor="middle" font-size="11" fill="#1e3a8a">chunked</text><rect x="200" y="60" width="140" height="80" rx="10" fill="#ede9fe" stroke="#7c3aed"/><text x="270" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">Embed</text><text x="270" y="110" text-anchor="middle" font-size="11" fill="#5b21b6">OpenAI / Cohere</text><text x="270" y="125" text-anchor="middle" font-size="11" fill="#5b21b6">→ 1536-d</text><rect x="380" y="60" width="140" height="80" rx="10" fill="#fef3c7" stroke="#f59e0b"/><text x="450" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#78350f">Pinecone Index</text><text x="450" y="110" text-anchor="middle" font-size="11" fill="#78350f">serverless</text><text x="450" y="125" text-anchor="middle" font-size="11" fill="#78350f">+ namespace</text><rect x="560" y="60" width="140" height="80" rx="10" fill="#dcfce7" stroke="#15803d"/><text x="630" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#065f46">LLM Answer</text><text x="630" y="110" text-anchor="middle" font-size="11" fill="#065f46">context + Q</text><text x="630" y="125" text-anchor="middle" font-size="11" fill="#065f46">GPT-4 / Claude</text><text x="172" y="105" font-size="22" fill="#475569">→</text><text x="352" y="105" font-size="22" fill="#475569">→</text><text x="532" y="105" font-size="22" fill="#475569">→</text><text x="360" y="35" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">Pinecone RAG flow</text><text x="360" y="180" text-anchor="middle" font-size="11" fill="#475569">Pinecone is fully managed — no servers, no Docker, scales by usage</text></svg>""")

## 1️⃣ Create a serverless index (illustrative)

```python
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="YOUR_KEY")

pc.create_index(
    name="company-docs",
    dimension=1536,                   # OpenAI text-embedding-3-small
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
index = pc.Index("company-docs")
```

Serverless indexes auto-scale storage and compute — you pay only for what you use.

## 2️⃣ Chunk documents (illustrative)

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = []
for doc in load_company_docs():       # PDFs, Confluence, etc.
    for i, text in enumerate(splitter.split_text(doc.text)):
        chunks.append({
            "id":      f"{doc.id}-{i}",
            "text":    text,
            "source":  doc.source,
            "team":    doc.team,
        })
```

Chunk size matters — too small loses context, too large dilutes the match. 300–800 tokens is typical.

## 3️⃣ Embed + upsert in batches (illustrative)

```python
from openai import OpenAI
client = OpenAI()

def embed(texts):
    resp = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [d.embedding for d in resp.data]

BATCH = 100
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i+BATCH]
    vecs = embed([c["text"] for c in batch])
    index.upsert(
        vectors=[
            {"id": c["id"], "values": v,
             "metadata": {"text": c["text"], "source": c["source"], "team": c["team"]}}
            for c, v in zip(batch, vecs)
        ],
        namespace="acme-corp",        # 🔑 multi-tenant isolation
    )
```

## 4️⃣ Retrieve top-K with metadata filtering (illustrative)

```python
def retrieve(question, team=None, k=5):
    qv = embed([question])[0]
    res = index.query(
        vector=qv,
        top_k=k,
        namespace="acme-corp",
        filter={"team": team} if team else None,
        include_metadata=True,
    )
    return [m["metadata"]["text"] for m in res["matches"]]

context = retrieve("what is our parental leave policy?", team="HR")
```

The `filter` field accepts MongoDB-style operators (`$eq`, `$in`, `$gte`, `$and`, `$or`).

## 5️⃣ Compose the prompt and call the LLM (illustrative)

```python
def answer(question, team=None):
    docs = retrieve(question, team=team, k=5)
    context = "\n\n---\n\n".join(docs)
    prompt = f"""You are a helpful assistant. Use ONLY the context below to answer.
If the answer isn't in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
ANSWER:"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

print(answer("how many sick days do I get?", team="HR"))
```

This is the core of every RAG product on the market today.

## 6️⃣ Production patterns

- **Re-ranking** — over-fetch top-50 from Pinecone, then rerank with a cross-encoder (Cohere Rerank, BGE Reranker) and keep top-5.
- **Hybrid search** — Pinecone supports **sparse + dense** hybrid (BM25-like sparse vectors mixed with dense).
- **Namespaces** for multi-tenant SaaS — one index, many isolated logical "tables".
- **Backups** — `index.create_collection()` snapshots; restore into a new index.
- **Cost control** — drop unused namespaces; serverless billing is per active vector + per query.

## ⚖️ Serverless vs Pod-based

| | Serverless | Pod-based |
|--|--|--|
| Pricing | Per use (vectors stored + queries) | Per pod-hour |
| Scaling | Automatic | Manual (resize pods) |
| Cold-start | Slight latency on first query | None |
| Best for | Variable / spiky traffic, dev | Predictable high QPS |

## 🎓 Recap — When to use Pinecone
✅ You want zero-ops, pure managed vector DB
✅ Building production RAG with metadata filters + namespaces
✅ Need quick deployment with no infra team
❌ Cost-sensitive, on-prem, or want open-source → use Qdrant / Milvus / pgvector

## 📚 Further reading
- https://docs.pinecone.io
- https://www.pinecone.io/learn/  (excellent free RAG course)